## Usando o pandas para analisar o arquivo
---
* Caminho: C:\Users\joao.victor\MSGÁS\GEOP - Documentos\2026\Manutenção - 2026

In [ ]:
#instala as bibliotecas

#!pip install pandas
#!pip install openpyxl
#!pip install dotenv

In [1]:
#importa bibliotecas
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv


In [2]:
load_dotenv()

caminho = os.getenv("FILE_BASE_DATA")
try:
    # Dataframe de horas trabalhas e viagem
    df = pd.read_excel(caminho,
                       sheet_name='IFS_TASK_CLOCKING',
                       usecols=[
                            "TASK_SEQ",
                            "TASK_DESCRIPTION",
                            "CLOCKING_CATEGORY",
                            "CLOCKING_TYPE",
                            "START_TIME",
                            "STOP_TIME",
                            "WORK_HOURS",
                            "ORGANIZATION_ID",
                            "EMPLOYEE_ID"
                       ]
    )
    print("Carregado dataframe")
except Exception as e:
    print('Erro ao ler o excel', e)

Carregado dataframe


## Tratamento de dados
1. Altera o nome das colunas para nomes intuitivos
   * N° Tarefa - TASK_SEQ
   * Descr da Tarefa - TASK_DESCRIPTION
   * Categoria de Registro - CLOCKING_CATEGORY (Viagem ou serviço)
   * Tipo de Reg de Horas - CLOCKING_TYPE (Registro de saída ou de entrada)
   * Org de Manut - ORGANIZATION_ID
   * Horário de Início - START_TIME
   * Hora Parada - STOP_TIME
   * Horas Trab - WORK_HOURS
   * ID Recurso - EMPLOYEE_ID
2. Limpa linhas que estão incompletas
3. Escolhe apenas serviços primarios e não analisa serviços de terceiros
4. Modifica os dtypes das colunas
5. Criar coluna "In Loco" para __serviços que nao tem viagem__
    1. Café da firma
    2. Troca de sobreaviso%
    2. Patrulha%
    2. Calibrar esta%
    2. Calibrar torr%
    2. Consolidação de clientes
    2. Palestra%
    2. EQS - Momento de segurança com GESMA
    2. Transferência de odorante e recarga de tanque da odorizadora
    2. Treinamento%
    2. Rest

In [3]:
# 1. mudando os nomes
df.rename(columns={"TASK_SEQ" : "ID_tarefa",
  "TASK_DESCRIPTION": "Descricao",
  "CLOCKING_CATEGORY": "Tipo_temporal",
  "CLOCKING_TYPE": "Valida_registro",
  "START_TIME": "Tempo_inicio",
  "STOP_TIME": "Tempo_fim",
  "WORK_HOURS": "Horas_trabalhadas",
  "ORGANIZATION_ID": "ORG_manut",
  "EMPLOYEE_ID": "TOM"}, inplace=True)

In [4]:
# 2. Retirando dados que estão em execucao/Incompletos
df.dropna(axis=0, how='any', inplace=True)

In [5]:
# 3. Retira serviços de terceiros ORG MANUT -> != MCGR, OCGR, TLG
filtroMCGR = df["ORG_manut"] == "MCGR"
filtroOCGR = df["ORG_manut"] == "OCGR"
filtroTLG = df["ORG_manut"] == "TLG"
df = df[filtroMCGR | filtroTLG | filtroOCGR]

In [6]:
# 4. Formata os tempo_inicio e tempo_fim
#       função de formatação
def formata_data(d):
    #Variaveis
    r = str(d) #Variavel de retorno
    month_number = {
        "jan": "01",
        "feb": "02",
        "mar": "03",
        "apr": "04",
        "may": "05",
        "jun": "06",
        "jul": "07",
        "aug": "08",
        "sep": "09",
        "oct": "10",
        "nov": "11",
        "dec": "12"
    } # Dicionario conversor de mes de nome para numero
    r = r.strip().lower().split() #tirando espaços e deoxando tudo minusculo
    day = r[1].replace(",", " ").strip()
    month = month_number[r[0]]
    year = r[2][:4]
    hour = pd.to_timedelta(r[3]) + pd.to_timedelta("12:00:00") if r[4] == "pm" else pd.to_timedelta(r[3]) #horas "brasileiras"
    hour = str(hour)[-8:] # Transform o dado em string
    #Formato antes: May 5, 2026, 2:37:30 PM
    #Formato que deve ficar 05/05/2026 02:37:30
    return day + "/" + month + "/" + year + " " + hour


# APLICA FUNÇÃO DE FORMATAÇÃO
df["Tempo_inicio"] = df["Tempo_inicio"].apply(formata_data)
df["Tempo_fim"] = df["Tempo_fim"].apply(formata_data)

# Colunas de tempo para deltatime
df["Tempo_inicio"] = pd.to_datetime(df["Tempo_inicio"], format="%d/%m/%Y %H:%M:%S")
df["Tempo_fim"] = pd.to_datetime(df["Tempo_fim"], format="%d/%m/%Y %H:%M:%S")

In [7]:
# 5. Adiciona coluna "In Loco"

#Toda descrição que contenha:
no_travel = [
    "Caf",
    "Troca de sobreavi",
    "Patru",
    "Calibrar esta",
    "Calibrar torr",
    "Preparar esta",
    "Consolida",
    "Palestra",
    "EQS",
    "recarga de tanque",
    "Treinamento",
    "test"
]
#Regex se tiver algo aqui dentro
regex = "|".join(no_travel)

#Cria Coluna
df["In Loco"] = df["Descricao"].str.contains(regex, case=False, na=False) #case ignora letras maiusculas ou minusculas analisa a palavra em si

In [ ]:
#Saida do excel limpo para melhor visualização
df.to_excel(r".\Data\teste.xlsx", index=False)

## Analisando dados
1. Escolhe um perído para analisar
1. O serviço realizado teve viajem?
    2. Serviço não precisa de viagem? (Apontamento correto)
    2. Serviço precisa de viagem:
        2. O tempo de viajem foi maior que 4min?(Apontamento correto)
        2. O tempo de viajem foi maior que 8hr? (Apontamento errado)
3. Acompanhamento pessoal

In [8]:
# 1. Seleciona um periodo para analisar
filtro_temporal = df["Tempo_inicio"].between(pd.to_datetime("05/01/2026"), pd.to_datetime("05/31/2026")) #Mês de maio
df_periodo = df[filtro_temporal].copy()
#   1.1 Reseta os indices para esse periodo
df_periodo.reset_index(drop=True, inplace=True)
df_periodo

,ID_tarefa,Tempo_inicio,Tipo_temporal,Valida_registro,TOM,Tempo_fim,ORG_manut,Descricao,Horas_trabalhadas,In Loco
0,206275,2026-05-30 09:47:28,Viagem,Saída Registrada,GUSTAVOG,2026-05-30 09:55:15,OCGR,LER MEDIDOR - JBS,0.1297,False
1,206275,2026-05-30 09:55:15,Serviço,Saída Registrada,GUSTAVOG,2026-05-30 09:57:16,OCGR,LER MEDIDOR - JBS,0.0336,False
2,206260,2026-05-29 16:04:01,Serviço,Saída Registrada,GILMARB,2026-05-29 17:13:59,MCGR,PREPARAR ESTAÇÃO SEARA,1.1661,True
3,206174,2026-05-30 07:06:28,Viagem,Saída Registrada,MERIVAN,2026-05-30 07:32:41,TLG,TLG1-VE-02 - Inspeção rotineira da Estação,0.4369,False
4,206174,2026-05-30 07:32:41,Serviço,Saída Registrada,MERIVAN,2026-05-30 07:33:56,TLG,TLG1-VE-02 - Inspeção rotineira da Estação,0.0208,False
...,...,...,...,...,...,...,...,...,...,...
2315,202478,2026-05-11 14:06:56,Viagem,Saída Registrada,CLAUDENIS,2026-05-11 15:30:17,OCGR,Inspeção de aterramento,1.3892,False
2316,202478,2026-05-11 15:30:17,Serviço,Saída Registrada,CLAUDENIS,2026-05-11 16:40:36,OCGR,Inspeção de aterramento,1.1719,False
2317,202408,2026-05-29 00:26:30,Serviço,Saída Registrada,SERGIO,2026-05-29 00:27:35,OCGR,Inspeção completa - Carretinha,0.0181,False
2318,202408,2026-05-29 10:05:07,Viagem,Saída Registrada,SERGIO,2026-05-29 10:05:10,OCGR,Inspeção completa - Carretinha,0.0008,False


In [39]:
# 2. SERVIÇO TEVE VIAGEM
#   Criação de filtros
filtro_Viagem = df_periodo["Tipo_temporal"] == "Viagem" # Linhas de viagem
filtro_Serviço = df_periodo["Tipo_temporal"] == "Serviço" # Linhas de Serviço

#   DF de Viagens e Serviço
Serviços = df_periodo[filtro_Serviço]
Viagens = df_periodo[filtro_Viagem]

#   Filtros para viagem
filtro_longa = Viagens["Horas_trabalhadas"] > 8
filtro_curta = Viagens["Horas_trabalhadas"] < 4/60
filtro_viagem_OK = ~filtro_curta & ~filtro_longa # Todas viagens que não estiverem gasto nem pouco e nem muito tempo -> DF Viagens
# Filtro viagem OK contempla todos serviços que tem o tempo de viagem compativel
# Porêm tem as viagens in loco (que não existe tempo de viagem), que possui TRUE na coluna "In Loco"
# Logo para de fato pegar todas a viagens OK é preciso fazer uma união "OR" dessa coluna com o filtro_viagem_OK
filtro_viagem_OK = filtro_viagem_OK | np.array(Viagens["In Loco"])

#   numeros das tarefas de viagem e serviço
tarefas_viagem = np.array(Viagens["ID_tarefa"], dtype=np.int32)
tarefas_Serviço = np.array(Serviços["ID_tarefa"], dtype=np.int32)

#   Filtro para Serviços sem viagem
filtro_serviço_com_viagem = np.isin(tarefas_Serviço, tarefas_viagem) # Linhas de serviço sem viagem -> DF Serviços
# Pega a tabela serviço e cria uma coluna viagem
Serviços["Viagem"] = filtro_serviço_com_viagem
Serviços["Viagem"] = Serviços["Viagem"] | Serviços["In Loco"]

In [42]:
# 3. Cria e exporta tabela que será utilizada para a criação dos gráficos
#   3.1 Cria coluna Valida serviço
buff = filtro_viagem_OK.reindex(range(0, len(df_periodo)), fill_value=False) #Expande o filtro para ficar do tamanho do df_periodo
buff = buff | Serviços["Viagem"].reindex(range(0, len(df_periodo)), fill_value=False)
df_periodo["Valida serviço"] = buff

#   3.2 Cria coluna Motivo
def insert_motivo(x):
    motivo = ""
    #       Se for um False na coluna serviço mostra o porque está errado
    if not x["Valida serviço"]:
        if x["Tipo_temporal"] == "Viagem":
        #           Se é uma viagem é porque o tempo está fora do esperado
            motivo = "Viagem curta ou longa"
        else:
        #          Se é um serviço como False significa que o serviço não tem viagem
            motivo = "Serviço sem viagem"
    return motivo

df_periodo["Motivo inconsistente"] = df_periodo.apply(insert_motivo, axis=1)


In [52]:
#_{df_periodo["Tempo_inicio"].max()}_a_{df_periodo["Tempo_inicio"].min()}
nome_arquivo = r".\data\Resultado_analise.xlsx"
df_periodo.to_excel(nome_arquivo,
                    columns=[
                        "ID_tarefa",
                        "Tempo_inicio",
                        "TOM",
                        "ORG_manut",
                        "Descricao",
                        "Horas_trabalhadas",
                        "Valida serviço",
                        "Motivo inconsistente",
                    ],
                    index=False)